# Uber Eats 商家流失预测｜Part 1：Prepare Data Signals

这一部分从五张原始业务表出发，先检查数据粒度和主键，再使用 SQL 构建 `merchant_id × cutoff_date` 粒度的 Feature Mart。完成后，我们会得到后续 EDA 和模型训练需要的一张完整宽表。

## 0. 本节目录

1. 读取五张原始业务表；
2. 检查每张表的粒度和主键；
3. 将 DataFrame 注册为 SQL 表；
4. 用 SQL 聚合订单、经营、客服和促销信号；
5. 连接各类信号，生成 Feature Mart；
6. 查看最终字段数量、样本数量和主键唯一性。

In [1]:
from pathlib import Path
import sqlite3
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# 仓库中的五张原始表统一保存在 data/。
DATA_DIR = Path("data")

# Part 1｜Prepare Data Signals

## 从五张原始表到 SQL Feature Mart

这一部分先明确每张原始表的粒度和主键，再通过 SQL 聚合不同时间窗口的业务信号，最终生成一行一个 `merchant_id × cutoff_date` 的 Feature Mart。

## 1. 读取五张原始表

### 为什么先看原始表？

真实项目中，字段往往来自不同系统。建模前必须知道每张表的 **grain（每一行代表什么）** 和 **key（什么字段应该唯一）**。

- `merchant_dim`：一行一个商家；
- `merchant_daily_sample`：一行一个商家一天；
- `orders_sample`：一行一笔订单；
- `support_fact`：一行一张客服工单；
- `promotion_fact`：一行一次促销活动。

如果直接把订单表、客服表和促销表连接起来，可能形成 many-to-many join，导致订单金额和工单数被重复计算。因此我们会先分别聚合，再在商家与 cutoff 粒度连接。

In [2]:
# 本次分析需要读取的五张业务表。
RAW_SHEETS = {'merchant_dim': 'merchant_dim.csv', 'merchant_daily': 'merchant_daily_sample.csv', 'orders': 'orders_sample.csv', 'support': 'support_fact.csv', 'promotion': 'promotion_fact.csv'}

In [3]:
def load_raw_tables(data_dir: Path) -> dict[str, pd.DataFrame]:
    """读取 V3 的五张原始业务表；标签也由这些表现场计算。"""
    if not data_dir.is_dir():
        raise FileNotFoundError(f"V3 data directory not found: {data_dir.resolve()}")
    tables: dict[str, pd.DataFrame] = {}
    for sql_name, file_name in RAW_SHEETS.items():
        file_path = data_dir / file_name
        if not file_path.exists():
            raise FileNotFoundError(f"Missing V3 raw table: {file_path.resolve()}")
        tables[sql_name] = pd.read_csv(file_path)
    return tables

In [4]:
raw_tables = load_raw_tables(DATA_DIR)

raw_summary = pd.DataFrame([
    {
        "table": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "unique_merchants": frame["merchant_id"].nunique(),
    }
    for name, frame in raw_tables.items()
])
raw_summary

,table,rows,columns,unique_merchants
0,merchant_dim,2000,9,2000
1,merchant_daily,240000,12,2000
2,orders,105095,25,1998
3,support,5492,8,1849
4,promotion,2687,8,1494


## 2. 检查主键，再注册 SQL 表

### 为什么检查主键？

主键重复通常意味着两类问题：上游数据真的重复，或我们误解了表的粒度。无论哪一种，都会让聚合指标偏大。

这里把 Pandas DataFrame 注册到内存中的 SQLite：用 Python 控制分析流程，用 SQL 完成业务特征聚合。

In [5]:
def register_sql_tables(
    raw_tables: dict[str, pd.DataFrame],
) -> tuple[sqlite3.Connection, pd.DataFrame]:
    """检查主键并将 DataFrame 注册为 SQLite 表。"""
    key_map = {
        "merchant_dim": "merchant_id",
        "orders": "order_id",
        "support": "ticket_id",
        "promotion": "promotion_id",
    }
    key_quality = pd.DataFrame(
        [
            {
                "table": table,
                "key": key,
                "rows": len(raw_tables[table]),
                "duplicate_rows": int(raw_tables[table].duplicated(key).sum()),
            }
            for table, key in key_map.items()
        ]
    )
    non_order_duplicates = key_quality.loc[
        key_quality["table"] != "orders", "duplicate_rows"
    ]
    if non_order_duplicates.gt(0).any():
        raise ValueError("Unexpected duplicate primary keys outside orders table")

    connection = sqlite3.connect(":memory:")
    for name, dataframe in raw_tables.items():
        sql_ready = dataframe.copy()
        if name == "orders":
            sql_ready = sql_ready.drop_duplicates("order_id", keep="first")
        for column in sql_ready.columns:
            if "date" in column or column in {"created_at", "resolved_at"}:
                sql_ready[column] = pd.to_datetime(
                    sql_ready[column], errors="coerce"
                ).astype("string")
        sql_ready.to_sql(name, connection, index=False, if_exists="replace")
    return connection, key_quality

In [6]:
connection, key_quality = register_sql_tables(raw_tables)
key_quality

,table,key,rows,duplicate_rows
0,merchant_dim,merchant_id,2000,0
1,orders,order_id,105095,0
2,support,ticket_id,5492,0
3,promotion,promotion_id,2687,0


## 3. 用 SQL 构建 Feature Mart

### 一行数据代表什么？

Feature Mart 的 grain 是：

> 一个商家在一个 `cutoff_date` 时点的历史状态。

所有模型特征只能使用 cutoff 之前的数据；标签使用 cutoff 之后 28 天的数据：

- Feature window：主要观察 cutoff 前 7、28 或 30 天；
- Label window：观察 `[cutoff_date, cutoff_date + 28 days)`；
- `future_orders_28d = 0` 时，`churn_label = 1`。

下面直接展开完整 SQL。SQL 会分别聚合订单、商家日状态、客服和促销数据，再连接成 Feature Mart。课堂上可以按 `cohorts → base → 各主题聚合 → 最终 SELECT` 的顺序逐段查看。

In [7]:
# 每个 cutoff_date 都会生成一份当时可用的商家特征快照。
CUTOFF_DATES = ['2026-03-01', '2026-04-01', '2026-05-01', '2026-06-01']

In [8]:
def build_feature_mart(connection: sqlite3.Connection) -> pd.DataFrame:
    """用 SQL 构建 merchant × cutoff 粒度的 Feature Mart。"""
    cohort_sql = " UNION ALL ".join(
        f"SELECT DATE('{cutoff}') AS cutoff_date" for cutoff in CUTOFF_DATES
    )
    feature_sql = f"""
    WITH cohorts AS (
        {cohort_sql}
    ),
    base AS (
        SELECT
            m.merchant_id,
            c.cutoff_date,
            STRFTIME('%Y-%m', c.cutoff_date) AS cohort,
            m.city,
            m.cuisine_type,
            m.segment,
            m.menu_ready,
            m.competitor_presence,
            CAST(JULIANDAY(c.cutoff_date) - JULIANDAY(m.join_date) AS INTEGER)
                AS account_age_days
        FROM merchant_dim AS m
        CROSS JOIN cohorts AS c
    ),
    order_raw AS (
        SELECT
            b.merchant_id,
            b.cutoff_date,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) >= DATE(b.cutoff_date, '-7 day')
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN 1 ELSE 0 END) AS weekly_orders_7d,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) >= DATE(b.cutoff_date, '-28 day')
                      AND DATE(o.order_date) < DATE(b.cutoff_date, '-7 day')
                     THEN 1 ELSE 0 END) AS previous_orders_21d,
            MAX(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN DATE(o.order_date) END) AS last_order_date,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN 1 ELSE 0 END) AS completed_orders_28d,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN o.gross_sales ELSE 0 END) AS gmv_28d,
            COUNT(DISTINCT CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN o.customer_id END) AS unique_customers_28d,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.new_customer_flag = 1
                     THEN 1 ELSE 0 END) AS new_orders_28d,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.repeat_customer_flag = 1
                     THEN 1 ELSE 0 END) AS repeat_orders_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.accepted_flag = 1 THEN 1 ELSE 0 END)
                AS accepted_orders_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.cancelled_flag = 1 THEN 1 ELSE 0 END)
                AS cancelled_orders_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.merchant_delay_flag = 1 THEN 1 ELSE 0 END)
                AS delayed_orders_28d,
            AVG(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN o.prep_time_minutes END) AS avg_prep_28d,
            AVG(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN o.prep_time_minutes * o.prep_time_minutes END)
                AS avg_prep_sq_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.late_delivery_flag = 1 THEN 1 ELSE 0 END)
                AS late_orders_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.refund_requested_flag = 1 THEN 1 ELSE 0 END)
                AS refund_orders_28d,
            AVG(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN o.customer_rating END) AS avg_rating_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.customer_rating = 1 THEN 1 ELSE 0 END)
                AS one_star_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.customer_rating IS NOT NULL THEN 1 ELSE 0 END)
                AS rated_orders_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.complaint_flag = 1 THEN 1 ELSE 0 END)
                AS complaints_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN COALESCE(o.commission_amount, 0) ELSE 0 END)
                AS commission_paid_28d,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN COALESCE(o.merchant_profit, 0) ELSE 0 END)
                AS profit_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.merchant_dispute_flag = 1 THEN 1 ELSE 0 END)
                AS disputes_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                      AND o.payment_failed_flag = 1 THEN 1 ELSE 0 END)
                AS failed_payments_28d,
            SUM(CASE WHEN DATE(o.order_date) < DATE(b.cutoff_date)
                     THEN 1 ELSE 0 END) AS all_orders_28d,
            SUM(CASE WHEN o.completed_flag = 1
                      AND DATE(o.order_date) >= DATE(b.cutoff_date)
                      AND DATE(o.order_date) < DATE(b.cutoff_date, '+28 day')
                     THEN 1 ELSE 0 END) AS future_orders_28d
        FROM base AS b
        LEFT JOIN orders AS o
          ON b.merchant_id = o.merchant_id
         AND DATE(o.order_date) >= DATE(b.cutoff_date, '-28 day')
         AND DATE(o.order_date) < DATE(b.cutoff_date, '+28 day')
        GROUP BY b.merchant_id, b.cutoff_date
    ),
    order_features AS (
        SELECT
            *,
            CASE WHEN previous_orders_21d > 0
                 THEN (weekly_orders_7d - previous_orders_21d / 3.0)
                      / (previous_orders_21d / 3.0) END
                AS order_trend_vs_prev3w,
            CASE WHEN last_order_date IS NOT NULL
                 THEN CAST(JULIANDAY(cutoff_date) - JULIANDAY(last_order_date)
                           AS INTEGER) END AS days_since_last_order,
            gmv_28d / NULLIF(completed_orders_28d, 0) AS aov_28d,
            new_orders_28d * 1.0 / NULLIF(completed_orders_28d, 0)
                AS new_customer_ratio_28d,
            cancelled_orders_28d * 1.0 / NULLIF(accepted_orders_28d, 0)
                AS cancellation_rate_28d,
            delayed_orders_28d * 1.0 / NULLIF(completed_orders_28d, 0)
                AS merchant_delay_rate_28d,
            SQRT(MAX(avg_prep_sq_28d - avg_prep_28d * avg_prep_28d, 0))
                AS prep_time_std_28d,
            late_orders_28d * 1.0 / NULLIF(completed_orders_28d, 0)
                AS late_delivery_rate_28d,
            refund_orders_28d * 1.0 / NULLIF(all_orders_28d, 0)
                AS refund_rate_28d,
            one_star_28d * 1.0 / NULLIF(rated_orders_28d, 0)
                AS one_star_review_rate_28d,
            complaints_28d * 1.0 / NULLIF(all_orders_28d, 0)
                AS complaint_rate_28d,
            repeat_orders_28d * 1.0 / NULLIF(completed_orders_28d, 0)
                AS repeat_customer_rate_28d,
            profit_28d / NULLIF(gmv_28d, 0) AS profit_margin_28d,
            disputes_28d * 1.0 / NULLIF(all_orders_28d, 0)
                AS dispute_rate_28d,
            failed_payments_28d * 1.0 / NULLIF(all_orders_28d, 0)
                AS failed_payment_rate_28d
        FROM order_raw
    ),
    daily_features AS (
        SELECT
            b.merchant_id,
            b.cutoff_date,
            SUM(CASE WHEN d.is_open = 1 THEN 1 ELSE 0 END)
                AS store_open_days_28d,
            AVG(d.business_hours_consistency)
                AS business_hours_consistency_28d,
            SUM(CASE WHEN d.temporary_closure_flag = 1 THEN 1 ELSE 0 END)
                AS temporary_closure_days_28d
        FROM base AS b
        LEFT JOIN merchant_daily AS d
          ON b.merchant_id = d.merchant_id
         AND DATE(d.date) >= DATE(b.cutoff_date, '-28 day')
         AND DATE(d.date) < DATE(b.cutoff_date)
        GROUP BY b.merchant_id, b.cutoff_date
    ),
    support_features AS (
        SELECT
            b.merchant_id,
            b.cutoff_date,
            COUNT(s.ticket_id) AS support_ticket_count_30d,
            AVG(CASE WHEN s.resolved_flag = 1 THEN s.resolution_hours END)
                AS issue_resolution_time_hours,
            SUM(CASE WHEN s.ticket_id IS NOT NULL
                      AND COALESCE(s.resolved_flag, 0) = 0 THEN 1 ELSE 0 END)
                * 1.0 / NULLIF(COUNT(s.ticket_id), 0)
                AS unresolved_ticket_rate_30d
        FROM base AS b
        LEFT JOIN support AS s
          ON b.merchant_id = s.merchant_id
         AND DATETIME(s.created_at) >= DATETIME(b.cutoff_date, '-30 day')
         AND DATETIME(s.created_at) < DATETIME(b.cutoff_date)
        GROUP BY b.merchant_id, b.cutoff_date
    ),
    promotion_features AS (
        SELECT
            b.merchant_id,
            b.cutoff_date,
            COUNT(p.promotion_id) AS promotion_participation_30d,
            SUM(COALESCE(p.promo_spend, 0)) AS promotion_spend_30d,
            (SUM(COALESCE(p.promotion_return, 0))
             - SUM(COALESCE(p.promo_spend, 0)))
             / NULLIF(SUM(COALESCE(p.promo_spend, 0)), 0)
                AS promotion_roi_30d
        FROM base AS b
        LEFT JOIN promotion AS p
          ON b.merchant_id = p.merchant_id
         AND DATE(p.start_date) < DATE(b.cutoff_date)
         AND DATE(p.end_date) >= DATE(b.cutoff_date, '-30 day')
        GROUP BY b.merchant_id, b.cutoff_date
    )
    SELECT
        b.merchant_id, b.cutoff_date, b.cohort,
        b.city, b.cuisine_type, b.segment,
        COALESCE(o.weekly_orders_7d, 0) AS weekly_orders_7d,
        o.order_trend_vs_prev3w,
        COALESCE(o.gmv_28d, 0) AS gmv_28d,
        o.aov_28d,
        COALESCE(o.unique_customers_28d, 0) AS unique_customers_28d,
        o.new_customer_ratio_28d,
        b.menu_ready,
        COALESCE(p.promotion_participation_30d, 0)
            AS promotion_participation_30d,
        COALESCE(d.store_open_days_28d, 0) AS store_open_days_28d,
        d.business_hours_consistency_28d,
        COALESCE(d.temporary_closure_days_28d, 0)
            AS temporary_closure_days_28d,
        o.cancellation_rate_28d,
        o.merchant_delay_rate_28d,
        o.avg_prep_28d AS avg_preparation_time_28d,
        o.prep_time_std_28d,
        o.late_delivery_rate_28d,
        o.refund_rate_28d,
        o.avg_rating_28d AS average_rating_28d,
        o.one_star_review_rate_28d,
        o.complaint_rate_28d,
        o.repeat_customer_rate_28d,
        COALESCE(o.commission_paid_28d, 0) AS commission_paid_28d,
        o.profit_margin_28d,
        p.promotion_spend_30d / NULLIF(o.gmv_28d, 0)
            AS promotion_cost_ratio_30d,
        p.promotion_roi_30d,
        o.dispute_rate_28d,
        COALESCE(s.support_ticket_count_30d, 0)
            AS support_ticket_count_30d,
        s.issue_resolution_time_hours,
        s.unresolved_ticket_rate_30d,
        o.failed_payment_rate_28d,
        b.account_age_days,
        b.competitor_presence,
        o.days_since_last_order,
        COALESCE(o.future_orders_28d, 0) AS future_orders_28d,
        CASE WHEN COALESCE(o.future_orders_28d, 0) = 0 THEN 1 ELSE 0 END
            AS churn_label
    FROM base AS b
    LEFT JOIN order_features AS o
      ON b.merchant_id = o.merchant_id AND b.cutoff_date = o.cutoff_date
    LEFT JOIN daily_features AS d
      ON b.merchant_id = d.merchant_id AND b.cutoff_date = d.cutoff_date
    LEFT JOIN support_features AS s
      ON b.merchant_id = s.merchant_id AND b.cutoff_date = s.cutoff_date
    LEFT JOIN promotion_features AS p
      ON b.merchant_id = p.merchant_id AND b.cutoff_date = p.cutoff_date
    ORDER BY b.cutoff_date, b.merchant_id
    """
    feature_mart = pd.read_sql_query(feature_sql, connection)
    feature_mart["cutoff_date"] = pd.to_datetime(feature_mart["cutoff_date"])
    return feature_mart

In [9]:
try:
    feature_mart = build_feature_mart(connection)
finally:
    connection.close()

print("Feature Mart shape:", feature_mart.shape)
feature_mart.head()

Feature Mart shape: (8000, 41)


,merchant_id,cutoff_date,cohort,city,cuisine_type,segment,weekly_orders_7d,order_trend_vs_prev3w,gmv_28d,aov_28d,unique_customers_28d,new_customer_ratio_28d,menu_ready,promotion_participation_30d,store_open_days_28d,business_hours_consistency_28d,temporary_closure_days_28d,cancellation_rate_28d,merchant_delay_rate_28d,avg_preparation_time_28d,prep_time_std_28d,late_delivery_rate_28d,refund_rate_28d,average_rating_28d,one_star_review_rate_28d,complaint_rate_28d,repeat_customer_rate_28d,commission_paid_28d,profit_margin_28d,promotion_cost_ratio_30d,promotion_roi_30d,dispute_rate_28d,support_ticket_count_30d,issue_resolution_time_hours,unresolved_ticket_rate_30d,failed_payment_rate_28d,account_age_days,competitor_presence,days_since_last_order,future_orders_28d,churn_label
0,M00001,2026-03-01,2026-03,Oakland,Mexican,SMB,3,2.000,130.940,21.823,6,0.167,1,0,23,0.717,4,0.000,0.167,18.765,5.170,0.000,0.000,4.667,0.000,0.000,0.833,33.270,0.183,0.000,NaN,0.000,0,NaN,NaN,0.000,559,0,1.000,3,0
1,M00002,2026-03-01,2026-03,San Jose,American,SMB,0,-1.000,125.430,25.086,5,0.000,1,0,24,0.692,3,0.000,0.200,24.228,3.335,0.000,0.000,3.800,0.000,0.000,1.000,29.670,0.310,0.000,NaN,0.000,1,26.350,0.000,0.000,599,0,11.000,6,0
2,M00003,2026-03-01,2026-03,San Francisco,Italian,SMB,2,2.000,119.210,29.803,4,0.750,1,0,22,0.616,2,0.000,0.000,24.785,1.808,0.000,0.000,4.750,0.000,0.250,0.250,30.870,0.142,0.000,NaN,0.000,0,NaN,NaN,0.000,796,1,1.000,5,0
3,M00004,2026-03-01,2026-03,Oakland,Japanese,Mid-Market,9,0.800,529.380,22.058,24,0.500,1,0,25,0.728,2,0.000,0.000,21.561,5.538,0.125,0.000,4.125,0.000,0.000,0.500,137.670,0.172,0.000,NaN,0.083,0,NaN,NaN,0.000,1822,1,1.000,11,0
4,M00005,2026-03-01,2026-03,San Francisco,Thai,SMB,2,2.000,104.790,26.197,4,0.250,1,0,25,0.752,2,0.000,0.000,21.008,6.583,0.000,0.250,4.750,0.000,0.000,0.750,27.250,0.187,0.000,NaN,0.000,0,NaN,NaN,0.000,1089,0,3.000,6,0
